# TextCNN 对照实验（TextCNN_Funning）

使用与 LoRA 实验一致的数据划分与超参数，输出统一指标格式。

In [ ]:
"""
第一部分：导入依赖
"""

import os
import time
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

from Bert_Config import CONFIG, setup_seed, load_raw_data, split_data, get_save_path

print("✅ 所有库导入完成")


✅ 所有库导入完成


In [ ]:

"""
第二部分：统一配置
"""

MODEL_NAME = "TextCNN"
DATA_PATH = CONFIG["DATA_DIR"]
RANDOM_SEED = CONFIG["RANDOM_SEED"]

# 从配置文件加载统一参数
MAX_VOCAB = CONFIG["CLASSIC_MAX_FEATURES"]
MAX_LEN = CONFIG["CLASSIC_MAX_LENGTH_TOKENS"]
EPOCHS = CONFIG["CLASSIC_EPOCHS"]
BATCH_SIZE = CONFIG["CLASSIC_BATCH_SIZE"]

# TextCNN模型特定参数（写死，不从CONFIG读取）
EMBED_DIM = 128  # 词嵌入维度
HIDDEN_DIM = 128  # 卷积核数量
LEARNING_RATE = 1e-3  # 学习率
CONV_KERNEL_SIZES = [3, 4, 5]  # 卷积核大小

SAVE_PATH = get_save_path("textcnn")

EXP_NAME = "TextCNN对照实验"
print("=" * 50)
print(f"实验配置: {EXP_NAME}")
print("=" * 50)
print(f"模型名称: {MODEL_NAME}")
print(f"数据目录: {DATA_PATH}")
print(f"随机种子: {RANDOM_SEED}")
print(f"最大词汇量: {MAX_VOCAB}")
print(f"最大文本长度: {MAX_LEN}")
print(f"词嵌入维度: {EMBED_DIM}")
print(f"卷积核数量: {HIDDEN_DIM}")
print(f"卷积核大小: [3, 4, 5]")
print(f"训练轮数: {EPOCHS}")
print(f"批次大小: {BATCH_SIZE}")
print(f"学习率: {LEARNING_RATE}")
print(f"模型保存路径: {SAVE_PATH}")
print("=" * 50)

实验配置: TextCNN对照实验
模型名称: TextCNN
数据目录: ../waimai.csv
随机种子: 42
最大词汇量: 5000
最大文本长度: 128
词嵌入维度: 128
卷积核数量: 128
卷积核大小: [3, 4, 5]
训练轮数: 5
批次大小: 64
学习率: 0.001
模型保存路径: ./textcnn_checkpoint


In [ ]:

"""
第三部分：文本编码与数据集
"""


def build_vocab(texts, max_vocab):
    """
    目的：从训练文本中构建词汇表，将字符映射为数字ID，用于后续文本编码
    
    数据流向：
    输入：
      - texts: 文本列表（通常是训练集的文本）
      - max_vocab: 词汇表的最大容量
    输出：
      - vocab: 字典，字符到ID的映射
    
    操作过程：
    1. 使用Counter统计所有文本中每个字符的出现频率
    2. 选择出现频率最高的 max_vocab-2 个字符（保留2个位置给特殊标记）
    3. 创建词汇表字典，包含两个特殊标记：
       - "<PAD>": 0 (填充标记，用于补齐短文本)
       - "<UNK>": 1 (未知字符标记，用于处理未在词汇表中的字符)
    4. 将高频字符按频率从高到低依次分配ID（从2开始）
    5. 返回完整的词汇表字典
    
    应用场景：
    在数据准备阶段，基于训练集构建词汇表，确保高频字符能被模型识别
    """
    counter = Counter()
    for text in texts:
        counter.update(list(text))
    most_common = counter.most_common(max_vocab - 2)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for idx, (tok, _) in enumerate(most_common, start=2):
        vocab[tok] = idx
    return vocab


def encode_text(text, vocab, max_len):
    """
    目的：将原始文本转换为数字ID序列，便于模型处理
    
    数据流向：
    输入：
      - text: 原始文本字符串
      - vocab: 词汇表字典（字符到ID的映射）
      - max_len: 目标序列长度
    输出：
      - ids: 整数列表，长度为max_len
    
    操作过程：
    1. 将文本字符串转换为字符列表
    2. 截取前max_len个字符（如果文本过长）
    3. 将每个字符转换为对应的ID：
       - 如果字符在词汇表中，使用其ID
       - 如果字符不在词汇表中，使用"<UNK>"的ID（1）
    4. 如果ID序列长度小于max_len，在末尾添加"<PAD>"的ID（0）进行填充
    5. 返回固定长度的ID序列
    
    应用场景：
    在数据加载时，将每条文本编码为固定长度的ID序列，供模型输入
    """
    tokens = list(text)
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens[:max_len]]
    if len(ids) < max_len:
        ids.extend([vocab["<PAD>"]] * (max_len - len(ids)))
    return ids


class TextDataset(Dataset):
    """
    目的：将文本数据和标签封装为PyTorch数据集，支持批量加载和迭代
    
    数据流向：
    输入：文本列表、标签列表、词汇表、最大长度
    输出：每次迭代返回 (编码后的文本张量, 标签张量)
    """
    
    def __init__(self, texts, labels, vocab, max_len):
        """
        目的：初始化数据集，存储文本、标签和编码所需参数
        
        数据流向：
        输入：
          - texts: 文本列表
          - labels: 标签列表
          - vocab: 词汇表（用于编码）
          - max_len: 最大长度（用于统一文本长度）
        输出：无（保存为实例属性）
        
        操作过程：
        1. 保存文本列表
        2. 保存标签列表
        3. 保存词汇表（用于编码）
        4. 保存最大长度（用于统一文本长度）
        """
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        """
        目的：返回数据集的大小，供DataLoader使用
        
        数据流向：
        输入：无
        输出：数据集样本数量（整数）
        
        操作过程：
        直接返回标签列表的长度（每个标签对应一个样本）
        """
        return len(self.labels)

    def __getitem__(self, idx):
        """
        目的：根据索引获取单个样本，将文本编码为张量，标签转换为张量
        
        数据流向：
        输入：
          - idx: 样本索引
        输出：
          - (input_ids张量, label张量)
        
        操作过程：
        1. 根据索引idx获取对应的文本和标签
        2. 使用encode_text函数将文本转换为ID序列
        3. 将ID序列转换为PyTorch长整型张量
        4. 将标签转换为PyTorch长整型张量
        5. 返回文本张量和标签张量的元组
        """
        input_ids = encode_text(self.texts[idx], self.vocab, self.max_len)
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(
            self.labels[idx], dtype=torch.long
        )

print("✅ 数据集构建函数定义完成")

✅ 数据集构建函数定义完成


In [ ]:

"""
第四部分：模型定义
"""


class TextCNN(nn.Module):
    """
    目的：定义TextCNN文本分类模型，通过多尺度卷积核提取文本特征进行分类
    
    数据流向：
    输入：文本ID序列 (batch_size, seq_len) 
    -> 词嵌入层 -> 多尺度卷积层 -> 最大池化层 -> 拼接 -> Dropout -> 全连接层 
    -> 输出：类别概率 (batch_size, num_classes)
    """
    
    def __init__(self, vocab_size, embed_dim, num_filters, num_classes=2):
        """
        目的：初始化TextCNN模型各层组件，定义网络结构
        
        数据流向：
        输入：
          - vocab_size: 词汇表大小
          - embed_dim: 词向量维度
          - num_filters: 每个卷积核的输出通道数（卷积核数量）
          - num_classes: 分类类别数（默认2）
        输出：无（初始化模型参数）
        
        操作过程：
        1. 调用父类初始化方法
        2. 创建词嵌入层：将字符ID映射为embed_dim维的向量
           - padding_idx=0 表示ID为0（填充标记）的向量始终为0
        3. 创建多个不同尺寸的卷积层（3-gram, 4-gram, 5-gram）：
           - 每个卷积核从不同的局部窗口提取特征
           - 输入通道数为embed_dim，输出通道数为num_filters
           - kernel_size分别为3、4、5，捕获不同长度的局部模式
        4. 创建Dropout层：防止过拟合，随机丢弃20%的神经元
        5. 创建全连接层：将拼接后的特征（num_filters * 3维）映射到类别数
        
        模型特点：
        多尺度卷积能够同时捕获短语、词组等不同粒度的文本特征
        """
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList(
            [
                nn.Conv1d(embed_dim, num_filters, kernel_size=k)
                for k in (3, 4, 5)
            ]
        )
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(num_filters * 3, num_classes)

    def forward(self, x):
        """
        目的：定义前向传播过程，将输入文本转换为分类结果
        
        数据流向：
        输入：x (batch_size, seq_len) - 文本ID序列
        -> embedding: (batch_size, seq_len, embed_dim) - 词向量序列
        -> transpose: (batch_size, embed_dim, seq_len) - 转置为卷积所需格式
        -> 多个卷积: 每个输出 (batch_size, num_filters, seq_len - kernel_size + 1)
        -> ReLU激活: 非线性变换
        -> 最大池化: 每个卷积输出池化为 (batch_size, num_filters) - 提取最重要特征
        -> 拼接: (batch_size, num_filters * 3) - 合并不同尺度的特征
        -> dropout: (batch_size, num_filters * 3) - 正则化
        -> fc: (batch_size, num_classes) - 类别logits
        
        操作过程：
        1. 通过词嵌入层将ID序列转换为词向量序列
        2. 转置维度，将(batch, seq, embed)转为(batch, embed, seq)，适配1D卷积
        3. 对每个卷积核进行卷积操作，并应用ReLU激活函数
        4. 对每个卷积结果进行最大池化，提取最显著的特征
        5. 将三个不同尺度的特征拼接成一个向量
        6. 应用Dropout进行正则化，防止过拟合
        7. 通过全连接层将特征映射到类别数，得到分类logits
        8. 返回分类结果（未经过softmax，用于计算交叉熵损失）
        """
        x = self.embedding(x).transpose(1, 2)
        convs = [torch.relu(conv(x)) for conv in self.convs]
        pools = [torch.max(c, dim=2).values for c in convs]
        out = torch.cat(pools, dim=1)
        out = self.dropout(out)
        return self.fc(out)

print("✅ 模型定义完成")

✅ 模型定义完成


In [ ]:

"""
第五部分：评估函数
"""


def evaluate(model, dataloader, criterion, device):
    """
    目的：评估模型在数据集上的性能，计算准确率、精确率、召回率、F1分数和平均损失
    
    数据流向：
    输入：
      - model: 训练好的模型
      - dataloader: 数据加载器（验证集或测试集）
      - criterion: 损失函数
      - device: 计算设备（CPU或GPU）
    输出：
      - (准确率, 精确率, 召回率, F1分数, 平均损失) 五元组
    
    操作过程：
    1. 将模型设置为评估模式（关闭dropout等训练时的特殊行为）
    2. 初始化累计损失和标签/预测列表
    3. 关闭梯度计算（节省内存和计算资源）
    4. 遍历数据加载器中的每个批次：
       a. 将数据移动到指定设备（CPU或GPU）
       b. 通过模型前向传播得到预测logits
       c. 计算批次损失并累加（乘以批次大小以计算总损失）
       d. 通过argmax获取预测类别（概率最大的类别）
       e. 将真实标签和预测标签收集到列表中
    5. 计算平均损失（总损失除以样本总数）
    6. 使用sklearn计算准确率、精确率、召回率、F1分数
    7. 返回所有评估指标
    
    应用场景：
    在每个epoch结束后评估模型在验证集上的性能，训练结束后评估测试集性能
    """
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            total_loss += loss.item() * batch_y.size(0)
            preds = logits.argmax(dim=1)
            all_labels.extend(batch_y.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    return acc, precision, recall, f1, avg_loss


print("✅ 评估函数定义完成")

✅ 评估函数定义完成


In [ ]:

"""
第六部分：准备数据
"""

setup_seed(RANDOM_SEED)
df = load_raw_data(DATA_PATH)
train_df, val_df, test_df = split_data(df, RANDOM_SEED)

vocab = build_vocab(train_df["review"], MAX_VOCAB)

train_dataset = TextDataset(
    train_df["review"].tolist(),
    train_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)
val_dataset = TextDataset(
    val_df["review"].tolist(),
    val_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)
test_dataset = TextDataset(
    test_df["review"].tolist(),
    test_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("✅ 数据准备完成")

✅ 数据准备完成


In [ ]:

"""
第七部分：模型保存函数
"""

def save_model(model, save_name):
    """
    目的：将训练好的模型参数保存到磁盘，便于后续加载和使用
    
    数据流向：
    输入：
      - model: 待保存的模型对象
      - save_name: 保存的文件名（如"best.pt"或"last.pt"）
    输出：无（模型参数保存到文件）
    
    操作过程：
    1. 检查保存目录是否存在，如果不存在则创建
    2. 拼接完整的保存路径（目录 + 文件名）
    3. 使用torch.save保存模型的state_dict（模型参数字典）
    4. 打印保存成功的消息和文件路径
    
    应用场景：
    训练过程中保存验证准确率最高的模型（best.pt），训练结束后保存最后一个epoch的模型（last.pt）
    """
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)
    save_file = os.path.join(SAVE_PATH, save_name)
    torch.save(model.state_dict(), save_file)
    print(f"✅ TextCNN模型已保存: {save_file}")

print("✅ 模型保存函数定义完成")

✅ 模型保存函数定义完成


In [ ]:

"""
第八部分：训练模型
"""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextCNN(len(vocab), EMBED_DIM, HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 添加学习率调度器：当验证准确率不再提升时降低学习率
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2, verbose=True
)

# 添加早停机制
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        
    def __call__(self, val_score):
        if self.best_score is None:
            self.best_score = val_score
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        else:
            self.best_score = val_score
            self.counter = 0
        return False

early_stopping = EarlyStopping(patience=3, min_delta=0.001)

print("\n开始训练...")
train_start_time = time.time()
best_dev_acc = 0

for epoch_num in range(EPOCHS):
    model.train()
    total_acc_train = 0
    total_loss_train = 0

    for batch_x, batch_y in tqdm(
        train_loader,
        desc=f"Epoch {epoch_num + 1}/{EPOCHS} [训练]",
    ):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        batch_loss = criterion(logits, batch_y)
        # 修复损失计算：batch_loss是平均损失，需要乘以批次大小得到总损失
        total_loss_train += batch_loss.item() * batch_y.size(0)

        acc = (logits.argmax(dim=1) == batch_y).sum().item()
        total_acc_train += acc

        optimizer.zero_grad()
        batch_loss.backward()
        # 添加梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    # 使用已定义的evaluate函数进行验证，获得完整的评估指标
    val_acc, val_precision, val_recall, val_f1, val_loss = evaluate(
        model, val_loader, criterion, device
    )
    
    # 计算训练集的平均损失和准确率
    train_loss_avg = total_loss_train / len(train_dataset)
    train_acc_avg = total_acc_train / len(train_dataset)
    
    print(
        f"[Epoch {epoch_num + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss_avg:.4f} | "
        f"Train Acc: {train_acc_avg:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Precision: {val_precision:.4f} | "
        f"Val Recall: {val_recall:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # 根据验证准确率更新学习率
    scheduler.step(val_acc)

    if val_acc > best_dev_acc:
        best_dev_acc = val_acc
        save_model(model, "best.pt")
        print(f"   🎯 发现更好的模型！验证准确率: {best_dev_acc:.3f}")

    # 检查早停
    if early_stopping(val_acc):
        print(f"   ⏹️  早停触发，验证准确率连续{early_stopping.patience}个epoch未提升")
        break

train_end_time = time.time()
training_time_sec = train_end_time - train_start_time
training_time_min = training_time_sec / 60

save_model(model, "last.pt")

print("\n✅ 训练完成！")
print(f"   - 最佳验证准确率: {best_dev_acc:.3f}")
print(f"   - 最佳模型已保存: {os.path.join(SAVE_PATH, 'best.pt')}")
print(f"   - 最后模型已保存: {os.path.join(SAVE_PATH, 'last.pt')}")


开始训练...


Epoch 1/5 [验证]: 100%|██████████| 19/19 [00:00<00:00, 59.44it/s]


[Epoch 1/5] Train Loss: 0.0058 | Train Acc: 0.8465 | Val Loss: 0.0048 | Val Acc: 0.8832
✅ TextCNN模型已保存: ./textcnn_checkpoint\best.pt
   🎯 发现更好的模型！验证准确率: 0.883


Epoch 2/5 [验证]: 100%|██████████| 19/19 [00:00<00:00, 52.67it/s]


[Epoch 2/5] Train Loss: 0.0036 | Train Acc: 0.9145 | Val Loss: 0.0046 | Val Acc: 0.8941
✅ TextCNN模型已保存: ./textcnn_checkpoint\best.pt
   🎯 发现更好的模型！验证准确率: 0.894


...

[Epoch 3/5] Train Loss: 0.0027 | Train Acc: 0.9381 | Val Loss: 0.0050 | Val Acc: 0.8882


Epoch 4/5 [验证]: 100%|██████████| 19/19 [00:01<00:00, 16.94it/s]


[Epoch 4/5] Train Loss: 0.0019 | Train Acc: 0.9605 | Val Loss: 0.0050 | Val Acc: 0.8841


Epoch 5/5 [验证]: 100%|██████████| 19/19 [00:01<00:00, 17.91it/s]

[Epoch 5/5] Train Loss: 0.0014 | Train Acc: 0.9705 | Val Loss: 0.0061 | Val Acc: 0.8874
✅ TextCNN模型已保存: ./textcnn_checkpoint\last.pt

✅ 训练完成！
   - 最佳验证准确率: 0.894
   - 最佳模型已保存: ./textcnn_checkpoint\best.pt
   - 最后模型已保存: ./textcnn_checkpoint\last.pt


In [ ]:

"""
第九部分：测试集评估
"""

# 加载最佳模型
print("\n加载最佳模型进行测试集评估...")
model.load_state_dict(torch.load(os.path.join(SAVE_PATH, "best.pt")))

# 使用已定义的evaluate函数进行测试集评估，确保评估逻辑一致
test_acc, test_precision, test_recall, test_f1, test_loss = evaluate(
    model, test_loader, criterion, device
)

print("\n测试集评估结果:")
print(f"  - Loss: {test_loss:.3f}")
print(f"  - Accuracy: {test_acc:.3f}")
print(f"  - Precision: {test_precision:.3f}")
print(f"  - Recall: {test_recall:.3f}")
print(f"  - F1 Score: {test_f1:.3f}")
print(f"   - 训练时间: {training_time_sec:.1f} 秒 ({training_time_min:.2f} 分钟)")

print(f"\n🎉 最终测试准确率: {test_acc:.3f}")


加载最佳模型进行测试集评估...


测试中: 100%|██████████| 19/19 [00:01<00:00, 17.28it/s]


测试集评估结果:
  - Loss:  0.005
  - Accuracy:  0.880
  - Precision:  0.868
  - Recall:  0.755
  - F1 Score:  0.807
   - 训练时间: 82.2 秒 (1.37 分钟)

🎉 最终测试准确率: 0.880
